## pix2pix

In [ ]:
import os

import torch
import torchvision.transforms as transforms
from datasets import load_dataset
from torchvision.transforms.functional import to_pil_image
from IPython.display import display
from PIL import Image

from models import Generator
from utils.hf_utils import ImageDataset

In [ ]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def load_checkpoint(checkpoint_path, device="cuda"):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model = Generator(
		img_size=checkpoint["img_size"],
		in_channels=checkpoint["generator_in_channels"],
	)
    model = model.to(device)
    model.load_state_dict(checkpoint["G"])
    return model

### Edges2Shoes

In [ ]:
edges2shoes_checkpoint_path = os.path.join("checkpoints", "edges2shoes", "edges2shoes_25.pth")

In [ ]:
edges2shoes_model = load_checkpoint(edges2shoes_checkpoint_path, device=device)

In [ ]:
hf_dataset = load_dataset("luethan2025/edges2shoes", split="val")
transform = transforms.Compose([
	transforms.ToTensor(),
])
edges_dataset = ImageDataset(hf_dataset, transform)

In [ ]:
random_idx = torch.randint(len(edges_dataset), (1,)).item()
edges_image, _ = edges_dataset[random_idx]
edges_image = edges_image.to(device)

edges_pil_image = to_pil_image(edges_image)
display(edges_pil_image)

In [ ]:
shoes_image = edges2shoes_model(edges_image.unsqueeze(0))
shoes_image = shoes_image.squeeze(0).detach().cpu()

shoes_image = (shoes_image + 1) / 2
shoes_image = shoes_image.clamp(0, 1)

shoes_pil_image = to_pil_image(shoes_image)
display(shoes_pil_image)

In [ ]:
edgesd_shoes_side_by_side_pil_image = Image.new("RGB", (edges_pil_image.width + shoes_pil_image.width, edges_pil_image.height))
edgesd_shoes_side_by_side_pil_image.paste(edges_pil_image, (0, 0))
edgesd_shoes_side_by_side_pil_image.paste(shoes_pil_image, (edges_pil_image.width, 0))
display(edgesd_shoes_side_by_side_pil_image)